# Table 1 — lapse-rate soil simulations, +1.2°C

Uses the existing `hrrrak_lapserate` and `met_hrrrak_lapserate` **soil-enabled** simulations at the original **+1.2°C precipitation threshold** for PPSA, TRAM, and HEEN. The original Table 1 calculations, observations, selected water years, averaging, rounding, and column layout are retained. Its 5 cm snow-presence criterion is separate from the precipitation-temperature threshold. Results display inline; no model runs or exports are generated.


In [1]:
# import packages 
%matplotlib inline

# plotting packages 
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns 
import matplotlib.dates as mdates

sns.set_theme()
plt.rcParams['figure.figsize'] = [12,6] #overriding size

# data packages 
import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime

import scipy
import os

In [2]:
pwd

'/home/cassie/python/repos/snow_modeling_point/sites/all_sites'

# Open Data and Model Simulations

## Function for Reading SMET Files 

In [3]:
def read_smet(filepath):
    header = {}
    fields = None
    data_start = None

    # Read file and parse header
    with open(filepath, "r") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        line = line.strip()

        # Detect fields line
        if line.startswith("fields"):
            fields = line.split("=")[1].strip().split()

        # Detect start of data
        if line == "[DATA]":
            data_start = i + 1
            break

        # Parse header key-value pairs
        if "=" in line and not line.startswith("["):
            key, value = line.split("=", 1)
            header[key.strip()] = value.strip()

    if fields is None:
        raise ValueError("No 'fields' line found in SMET header.")
    if data_start is None:
        raise ValueError("No [DATA] section found.")

    # Read data into DataFrame
    df = pd.read_csv(
        filepath,
        skiprows=data_start,
        sep=r"\s+",
        names=fields,
        parse_dates=["timestamp"]
    )

    # Set timestamp as index
    df = df.set_index("timestamp")

    # Convert to xarray
    ds = xr.Dataset.from_dataframe(df)

    return ds, header

### Open SNOWPACK SMet Output

In [4]:
# HRRR-AK Files First 

# HRRR-AK Lapse-rate Files 
ds_snowpack_hrrrak_lapserate_ppsa, header_hrrrak_lapserate_ppsa = read_smet("/home/cassie/python/models/run_snowpack/sites/ppsa/output/hrrrak_lapserate_ppsa_WY2020-WY2025_soil.smet")
ds_snowpack_hrrrak_lapserate_tram, header_hrrrak_lapserate_tram = read_smet("/home/cassie/python/models/run_snowpack/sites/tram/output/hrrrak_lapserate_tram_WY2020-WY2025_soil.smet")
ds_snowpack_hrrrak_lapserate_heen, header_hrrrak_lapserate_heen = read_smet("/home/cassie/python/models/run_snowpack/sites/heen/output/hrrrak_lapserate_heen_WY2020-WY2025_soil.smet")

# Met HRRR-AK Files now

# Met HRRR-AK Lapse-rate Files
ds_snowpack_met_hrrrak_lapserate_ppsa, header_met_hrrrak_lapserate_ppsa = read_smet("/home/cassie/python/models/run_snowpack/sites/ppsa/output/met_hrrrak_lapserate_ppsa_WY2020-WY2025_soil.smet") # ppsa has WY2020-WY2025
ds_snowpack_met_hrrrak_lapserate_tram, header_met_hrrrak_lapserate_tram = read_smet("/home/cassie/python/models/run_snowpack/sites/tram/output/met_hrrrak_lapserate_tram_WY2023-WY2025_soil.smet")
ds_snowpack_met_hrrrak_lapserate_heen, header_met_hrrrak_lapserate_heen = read_smet("/home/cassie/python/models/run_snowpack/sites/heen/output/met_hrrrak_lapserate_heen_WY2020-WY2022_soil.smet") # heen doesn't have until 2025


### Open Observations

In [5]:
# open observations

# HEEN 
file_heen = "/hdd/snow_hydrology/met_station/snotel/heenlatinee/heen_met_2016_2026_cleaned_v1.nc"
ds_obs_heen = xr.open_dataset(file_heen)
ds_obs_heen = ds_obs_heen.sel(time=slice("2019-10-01", "2025-09-30"))

#PPSA 
file_ppsa = "/hdd/snow_hydrology/met_station/ppsa2/pppsa_met_station_data_synoptic_2026-03-20" 
ds_obs_ppsa = xr.open_dataset(file_ppsa)
ds_obs_ppsa = ds_obs_ppsa.sel(time=slice("2019-10-01", "2025-09-30"))


# TRAM 
file_tram = "/hdd/snow_hydrology/met_station/tram/tram_met_station_data_synoptic_2026-03-20"  
ds_obs_tram = xr.open_dataset(file_tram)
ds_obs_tram = ds_obs_tram.sel(time=slice("2019-10-01", "2025-09-30"))

In [6]:
# pick colors for each site and stick to them across all plots
ppsa_color = 'darkviolet'
tram_color = 'maroon'
heen_color = 'teal'

# color hrrr-ak model 
hrrrak_color = 'tab:blue'
met_hrrrak_color = 'tab:green'
obs_color = 'tab:gray'

## Cut to just three years each

In [7]:
ds_obs_heen = ds_obs_heen.sel(time=slice("2019-10-01", "2022-09-30"))
ds_obs_ppsa = ds_obs_ppsa.sel(time=slice("2022-10-01", "2025-09-30"))
ds_obs_tram = ds_obs_tram.sel(time=slice("2022-10-01", "2025-09-30"))

# Statistics

#### List of everything: 

`RMSE`

> Overall snow depth performance

`Bias`
> Mean over/underprediction

`MAE`
> Robust average absolute error

`Peak Snow Depth Error`
> Magnitude error of seasonal max

`Snow Onset Error`
> Early/late snowpack establishment

`Peak Date Error`
> Timing of seasonal maximum

`Snow Disappearance Error`	
> Melt-out timing performance

`Melt Duration Error`
> Melt season representation

#### Most common in snow hydro papers, 
RMSE

Bias

Peak SWE/HS error

Peak timing error

Melt-out timing error


In [8]:
ds_snowpack_hrrrak_lapserate_ppsa


<xarray.Dataset> Size: 29MB
Dimensions:               (timestamp: 51883)
Coordinates:
  * timestamp             (timestamp) datetime64[ns] 415kB 2019-10-01T06:00:0...
Data variables: (12/68)
    Qs                    (timestamp) float64 415kB -209.9 -10.38 ... 21.66
    Ql                    (timestamp) float64 415kB 395.3 48.37 ... -82.42
    Qg                    (timestamp) float64 415kB 0.06 0.06 0.06 ... 0.06 0.06
    TSG                   (timestamp) float64 415kB 8.147 6.084 ... 13.03 12.99
    Qg0                   (timestamp) float64 415kB -131.5 -37.32 ... -11.14
    Qr                    (timestamp) float64 415kB -4.55 -0.098 ... 0.0 0.0
    ...                    ...
    zSs                   (timestamp) float64 415kB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    Ss                    (timestamp) float64 415kB 6.0 6.0 6.0 ... 6.0 6.0 6.0
    zS4                   (timestamp) float64 415kB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    S4                    (timestamp) float64 415kB 6.0 6.0 6.0 ... 6.0 6.0 6.0
    zS5                   (timestamp) float64 415kB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    S5                    (timestamp) float64 415kB 0.0 0.0 0.0 ... 0.0 0.0 0.0

In [9]:
# the model have a "timestamp" instead of "time" dimension, so rename that for consistency
# xarray rename returns a new dataset unless inplace=False, so assign the result.

ds_snowpack_hrrrak_lapserate_ppsa = ds_snowpack_hrrrak_lapserate_ppsa.rename({"timestamp": "time"})
ds_snowpack_hrrrak_lapserate_tram = ds_snowpack_hrrrak_lapserate_tram.rename({"timestamp": "time"})
ds_snowpack_hrrrak_lapserate_heen = ds_snowpack_hrrrak_lapserate_heen.rename({"timestamp": "time"})


ds_snowpack_met_hrrrak_lapserate_ppsa = ds_snowpack_met_hrrrak_lapserate_ppsa.rename({"timestamp": "time"})
ds_snowpack_met_hrrrak_lapserate_tram = ds_snowpack_met_hrrrak_lapserate_tram.rename({"timestamp": "time"})
ds_snowpack_met_hrrrak_lapserate_heen = ds_snowpack_met_hrrrak_lapserate_heen.rename({"timestamp": "time"})


In [10]:
import numpy as np
import pandas as pd
import xarray as xr

# ============================================================
# USER SETTINGS
# ============================================================

snow_threshold_cm = 5.0

# Water years to evaluate by site
site_water_years = {
    "PPSA": [2023, 2024, 2025],
    "TRAM": [2023, 2024, 2025],
    "HEEN": [2020, 2021, 2022],
}

# ============================================================
# Helper functions
# ============================================================

def water_year_slice(da, wy):
    """
    Select one water year.
    WY2025 = 2024-10-01 through 2025-09-30.
    """
    start = f"{wy-1}-10-01"
    end = f"{wy}-09-30 23:59:59"
    return da.sel(time=slice(start, end))


def align_model_obs(model_da, obs_da):
    """
    Align model and observation data on common valid timestamps.
    """
    model_df = model_da.to_dataframe(name="model")
    obs_df = obs_da.to_dataframe(name="obs")

    df = pd.concat([model_df, obs_df], axis=1).dropna()

    return df


def calculate_rmse(model, obs):
    return np.sqrt(np.nanmean((model - obs) ** 2))


def calculate_bias(model, obs):
    return np.nanmean(model - obs)


def calculate_peak_snow_depth_error(model, obs):
    return np.nanmax(model) - np.nanmax(obs)


def get_peak_snow_date(series):
    """
    Return date of maximum snow depth.
    """
    if series.dropna().empty:
        return pd.NaT

    return series.idxmax()


def get_snow_disappearance_date(series, threshold=5.0):
    """
    Snow disappearance is defined as the last timestamp in the water year
    when snow depth is greater than or equal to the threshold.

    This avoids false melt-out during temporary mid-winter melt periods.
    """
    snow_present = series >= threshold

    if not snow_present.any():
        return pd.NaT

    return series.index[snow_present][-1]


def get_ablation_duration_days(series, threshold=5.0):
    """
    Ablation duration is the time from peak snow depth to snow disappearance.
    """
    peak_date = get_peak_snow_date(series)
    disappearance_date = get_snow_disappearance_date(series, threshold)

    if pd.isna(peak_date) or pd.isna(disappearance_date):
        return np.nan

    return (disappearance_date - peak_date).days


def safe_date_error(model_date, obs_date):
    """
    Difference between modeled and observed dates in days.
    Positive = modeled event occurs later.
    Negative = modeled event occurs earlier.
    """
    if pd.isna(model_date) or pd.isna(obs_date):
        return np.nan

    return (model_date - obs_date).days


# ============================================================
# Compute statistics for one site/model/water year
# ============================================================

def compute_snowpack_statistics_by_wy(
    model_da,
    obs_da,
    model_name,
    site_name,
    wy,
    threshold=5.0
):
    # --------------------------------------------------------
    # Slice to water year
    # --------------------------------------------------------

    model_wy = water_year_slice(model_da, wy)
    obs_wy = water_year_slice(obs_da, wy)

    # --------------------------------------------------------
    # Align model and observations
    # --------------------------------------------------------

    df = align_model_obs(model_wy, obs_wy)

    if df.empty:
        return {
            "Site": site_name,
            "Water Year": f"WY{wy}",
            "Model": model_name,
            "RMSE (cm)": np.nan,
            "Bias (cm)": np.nan,
            "Peak Snow Depth Error (cm)": np.nan,
            "Snow Disappearance Error (days)": np.nan,
            "Ablation Duration Error (days)": np.nan,
            "Observed Peak Snow Depth (cm)": np.nan,
            "Modeled Peak Snow Depth (cm)": np.nan,
        }

    model = df["model"]
    obs = df["obs"]

    # --------------------------------------------------------
    # Magnitude metrics
    # --------------------------------------------------------

    rmse = calculate_rmse(model, obs)
    bias = calculate_bias(model, obs)
    peak_error = calculate_peak_snow_depth_error(model, obs)

    # --------------------------------------------------------
    # Timing metrics
    # --------------------------------------------------------

    model_disappearance = get_snow_disappearance_date(
        model,
        threshold=threshold
    )

    obs_disappearance = get_snow_disappearance_date(
        obs,
        threshold=threshold
    )

    disappearance_error = safe_date_error(
        model_disappearance,
        obs_disappearance
    )

    model_ablation_duration = get_ablation_duration_days(
        model,
        threshold=threshold
    )

    obs_ablation_duration = get_ablation_duration_days(
        obs,
        threshold=threshold
    )

    ablation_duration_error = (
        model_ablation_duration - obs_ablation_duration
    )

    # --------------------------------------------------------
    # Build result
    # --------------------------------------------------------

    return {
        "Site": site_name,
        "Water Year": f"WY{wy}",
        "Model": model_name,

        "RMSE (cm)": rmse,
        "Bias (cm)": bias,
        "Peak Snow Depth Error (cm)": peak_error,
        "Snow Disappearance Error (days)": disappearance_error,
        "Ablation Duration Error (days)": ablation_duration_error,

        "Observed Peak Snow Depth (cm)": np.nanmax(obs),
        "Modeled Peak Snow Depth (cm)": np.nanmax(model),
    }


# ============================================================
# Build list of FINAL manuscript comparisons
# ============================================================

comparisons = [

    # PPSA
    {
        "site": "PPSA",
        "model_name": "HRRR-AK Forcing",
        "model": ds_snowpack_hrrrak_lapserate_ppsa.HS_mod,
        "obs": ds_obs_ppsa["hs"],
    },
    {
        "site": "PPSA",
        "model_name": "Met Station Mixed Forcing",
        "model": ds_snowpack_met_hrrrak_lapserate_ppsa.HS_mod,
        "obs": ds_obs_ppsa["hs"],
    },

    # TRAM
    {
        "site": "TRAM",
        "model_name": "HRRR-AK Forcing",
        "model": ds_snowpack_hrrrak_lapserate_tram.HS_mod,
        "obs": ds_obs_tram["hs"],
    },
    {
        "site": "TRAM",
        "model_name": "Met Station Mixed Forcing",
        "model": ds_snowpack_met_hrrrak_lapserate_tram.HS_mod,
        "obs": ds_obs_tram["hs"],
    },

    # HEEN
    {
        "site": "HEEN",
        "model_name": "HRRR-AK Forcing",
        "model": ds_snowpack_hrrrak_lapserate_heen.HS_mod,
        "obs": ds_obs_heen["hs"],
    },
    {
        "site": "HEEN",
        "model_name": "Met Station Mixed Forcing",
        "model": ds_snowpack_met_hrrrak_lapserate_heen.HS_mod,
        "obs": ds_obs_heen["hs"],
    },
]


# ============================================================
# Calculate yearly statistics
# ============================================================

results = []

for comparison in comparisons:

    site = comparison["site"]

    for wy in site_water_years[site]:

        stats = compute_snowpack_statistics_by_wy(
            model_da=comparison["model"],
            obs_da=comparison["obs"],
            model_name=comparison["model_name"],
            site_name=site,
            wy=wy,
            threshold=snow_threshold_cm,
        )

        results.append(stats)

df_yearly = pd.DataFrame(results)

# ============================================================
# Add average rows for each Site / Model
# ============================================================

metric_cols = [
    "RMSE (cm)",
    "Bias (cm)",
    "Peak Snow Depth Error (cm)",
    "Snow Disappearance Error (days)",
    "Ablation Duration Error (days)",
    "Observed Peak Snow Depth (cm)",
    "Modeled Peak Snow Depth (cm)",
]

df_avg = (
    df_yearly
    .groupby(["Site", "Model"], as_index=False)[metric_cols]
    .mean()
)

df_avg["Water Year"] = "Average"

# Match column order
df_avg = df_avg[df_yearly.columns]

# Combine yearly and average rows
df_results = pd.concat(
    [df_yearly, df_avg],
    ignore_index=True
)

# ============================================================
# Sort table in desired order
# ============================================================

site_order = ["PPSA", "TRAM", "HEEN"]

model_order = [
    "HRRR-AK Forcing",
    "Met Station Mixed Forcing",
]

wy_order = {
    "WY2020": 2020,
    "WY2021": 2021,
    "WY2022": 2022,
    "WY2023": 2023,
    "WY2024": 2024,
    "WY2025": 2025,
    "Average": 9999,
}

df_results["Site_order"] = df_results["Site"].map(
    {site: i for i, site in enumerate(site_order)}
)

df_results["Model_order"] = df_results["Model"].map(
    {model: i for i, model in enumerate(model_order)}
)

df_results["WY_order"] = df_results["Water Year"].map(wy_order)

df_results = (
    df_results
    .sort_values(["Site_order", "WY_order", "Model_order"])
    .drop(columns=["Site_order", "Model_order", "WY_order"])
    .reset_index(drop=True)
)

# ============================================================
# Format output precision
# ============================================================

# USER SETTING:
# Change this to 1 or 2 depending on preferred displayed precision
sig_figs = 2

def round_sig(x, sig=2):
    """
    Round to a specified number of significant figures.
    Keeps NaN values as NaN.
    """
    if pd.isna(x):
        return np.nan
    if x == 0:
        return 0
    return round(x, sig - int(np.floor(np.log10(abs(x)))) - 1)


sig_fig_cols = [
    "RMSE (cm)",
    "Bias (cm)",
    "Peak Snow Depth Error (cm)",
    "Observed Peak Snow Depth (cm)",
    "Modeled Peak Snow Depth (cm)",
]

df_results[sig_fig_cols] = df_results[sig_fig_cols].map(
    lambda x: round_sig(x, sig=sig_figs)
)

day_cols = [
    "Snow Disappearance Error (days)",
    "Ablation Duration Error (days)",
]

# Keep day errors as whole days
df_results[day_cols] = df_results[day_cols].round(0)

# ============================================================
# Display results
# ============================================================

print("\n")
print("========================================================")
print("SNOWPACK MODEL PERFORMANCE SUMMARY")
print(f"Snow presence threshold = {snow_threshold_cm} cm")
print("========================================================")
print("\n")

print(df_results.to_string(index=False))

# ============================================================
# Separate tables for manuscript and supplement
# ============================================================

df_table1_average = df_results[
    df_results["Water Year"] == "Average"
].copy()

df_supplement_yearly = df_results[
    df_results["Water Year"] != "Average"
].copy()

print("\n")
print("========================================================")
print("TABLE 1: AVERAGE MODEL PERFORMANCE")
print("========================================================")
print("\n")
print(df_table1_average.to_string(index=False))

print("\n")
print("========================================================")
print("SUPPLEMENTAL TABLE: YEARLY MODEL PERFORMANCE")
print("========================================================")
print("\n")
print(df_supplement_yearly.to_string(index=False))

# ============================================================
# Optional: Save tables
# ============================================================

# df_results.to_csv(
#     "snowpack_model_performance_metrics_all_years_and_average.csv",
#     index=False
# )

# df_table1_average.to_csv(
#     "snowpack_model_performance_metrics_table1_average.csv",
#     index=False
# )

# df_supplement_yearly.to_csv(
#     "snowpack_model_performance_metrics_supplement_yearly.csv",
#     index=False
# )



SNOWPACK MODEL PERFORMANCE SUMMARY
Snow presence threshold = 5.0 cm


Site Water Year                     Model  RMSE (cm)  Bias (cm)  Peak Snow Depth Error (cm)  Snow Disappearance Error (days)  Ablation Duration Error (days)  Observed Peak Snow Depth (cm)  Modeled Peak Snow Depth (cm)
PPSA     WY2023           HRRR-AK Forcing       45.0       39.0                        45.0                              7.0                            16.0                          340.0                         390.0
PPSA     WY2023 Met Station Mixed Forcing       35.0       30.0                        47.0                              7.0                             8.0                          340.0                         390.0
PPSA     WY2024           HRRR-AK Forcing       28.0       17.0                         1.3                             12.0                            11.0                          270.0                         270.0
PPSA     WY2024 Met Station Mixed Forcing       23.0    

## Table 1 Only

In [11]:
# ============================================================
# TABLE 1 FOR MANUSCRIPT
# Final forcing configurations only
# ============================================================

table1 = df_table1_average[
    [
        "Site",
        "Model",
        "RMSE (cm)",
        "Bias (cm)",
        "Peak Snow Depth Error (cm)",
        "Snow Disappearance Error (days)",
        "Ablation Duration Error (days)",
    ]
].copy()

# Round manuscript values
table1 = table1.round(
    {
        "RMSE (cm)": 0,
        "Bias (cm)": 0,
        "Peak Snow Depth Error (cm)": 0,
        "Snow Disappearance Error (days)": 0,
        "Ablation Duration Error (days)": 0,
    }
)

print("\n")
print("========================================================")
print("TABLE 1: AVERAGE SNOWPACK MODEL PERFORMANCE")
print("========================================================")
print("\n")

print(table1.to_string(index=False))
from IPython.display import display
display(table1)




TABLE 1: AVERAGE SNOWPACK MODEL PERFORMANCE


Site                     Model  RMSE (cm)  Bias (cm)  Peak Snow Depth Error (cm)  Snow Disappearance Error (days)  Ablation Duration Error (days)
PPSA           HRRR-AK Forcing       34.0       25.0                        37.0                              6.0                            18.0
PPSA Met Station Mixed Forcing       26.0       13.0                        19.0                              4.0                             4.0
TRAM           HRRR-AK Forcing       28.0       15.0                        52.0                              5.0                            -5.0
TRAM Met Station Mixed Forcing       31.0      -18.0                       -33.0                             -8.0                             9.0
HEEN           HRRR-AK Forcing       23.0        2.0                        -2.0                             12.0                             7.0
HEEN Met Station Mixed Forcing       37.0       14.0                        

,Site,Model,RMSE (cm),Bias (cm),Peak Snow Depth Error (cm),Snow Disappearance Error (days),Ablation Duration Error (days)
6,PPSA,HRRR-AK Forcing,34.0,25.0,37.0,6.0,18.0
7,PPSA,Met Station Mixed Forcing,26.0,13.0,19.0,4.0,4.0
14,TRAM,HRRR-AK Forcing,28.0,15.0,52.0,5.0,-5.0
15,TRAM,Met Station Mixed Forcing,31.0,-18.0,-33.0,-8.0,9.0
22,HEEN,HRRR-AK Forcing,23.0,2.0,-2.0,12.0,7.0
23,HEEN,Met Station Mixed Forcing,37.0,14.0,41.0,15.0,49.0


## Final forcing comparison across sites

These numbers use the **unrounded yearly metrics** (`df_yearly`), excluding the already averaged Table 1 rows. Each paired site–water-year receives equal weight: three years at each of PPSA, TRAM, and HEEN, or nine seasons per forcing. The average RMSE is the arithmetic mean of seasonal RMSEs, not an RMSE pooled over all hourly residuals. The original selected periods, available observations, and partial-year coverage are retained.

Signed bias indicates overprediction (+) or underprediction (−). Absolute seasonal errors are averaged separately so opposite signs do not cancel when comparing peak-depth and timing performance. Each metric uses only site–years with valid values for both forcing datasets.


In [12]:
from IPython.display import display, Markdown

forcing_names = ["HRRR-AK Forcing", "Met Station Mixed Forcing"]
forcing_labels = {"HRRR-AK Forcing": "HRRR-AK", "Met Station Mixed Forcing": "Met Mixed"}
summary_definitions = {
    "Mean seasonal RMSE (cm)": ("RMSE (cm)", False),
    "Mean signed bias (cm)": ("Bias (cm)", False),
    "Mean absolute seasonal bias (cm)": ("Bias (cm)", True),
    "Mean absolute peak-depth error (cm)": ("Peak Snow Depth Error (cm)", True),
    "Mean absolute disappearance error (days)": ("Snow Disappearance Error (days)", True),
    "Mean absolute ablation-duration error (days)": ("Ablation Duration Error (days)", True),
}
forcing_summary = pd.DataFrame(index=forcing_names)
paired_metric_counts = {}
for label, (metric, use_absolute) in summary_definitions.items():
    paired = df_yearly.pivot(index=["Site", "Water Year"], columns="Model", values=metric)
    paired = paired.reindex(columns=forcing_names).dropna()
    paired_metric_counts[label] = len(paired)
    values = paired.abs() if use_absolute else paired
    forcing_summary[label] = values.mean().reindex(forcing_names)
forcing_summary.index = forcing_summary.index.map(forcing_labels)
forcing_summary.index.name = "Forcing"
display(forcing_summary.round(2))
display(pd.Series(paired_metric_counts, name="Paired site–years used").to_frame())

# Compare the same site–water-years using unrounded seasonal RMSEs.
paired_rmse = df_yearly.pivot(index=["Site", "Water Year"], columns="Model", values="RMSE (cm)")
paired_rmse = paired_rmse.reindex(columns=forcing_names).dropna()
hr_mean = paired_rmse[forcing_names[0]].mean()
met_mean = paired_rmse[forcing_names[1]].mean()
rmse_change = met_mean - hr_mean
rmse_pct_change = 100 * rmse_change / hr_mean if hr_mean != 0 else np.nan
met_lower = int((paired_rmse[forcing_names[1]] < paired_rmse[forcing_names[0]]).sum())
hr_lower = int((paired_rmse[forcing_names[0]] < paired_rmse[forcing_names[1]]).sum())
ties = len(paired_rmse) - met_lower - hr_lower

lines = [
    f"- Average seasonal snow-depth RMSE: **HRRR-AK {hr_mean:.2f} cm**; **Met Mixed {met_mean:.2f} cm**.",
    f"- Met Mixed − HRRR-AK average RMSE: **{rmse_change:+.2f} cm ({rmse_pct_change:+.1f}%)**.",
    f"- Met Mixed has lower RMSE in **{met_lower}/{len(paired_rmse)} site–years**; "
    f"HRRR-AK in **{hr_lower}/{len(paired_rmse)}**; **{ties} ties**.",
]
for forcing in ["HRRR-AK", "Met Mixed"]:
    row = forcing_summary.loc[forcing]
    lines.append(f"- **{forcing}:** mean signed bias **{row['Mean signed bias (cm)']:+.2f} cm**; "
        f"mean absolute peak-depth error **{row['Mean absolute peak-depth error (cm)']:.2f} cm**; "
        f"mean absolute disappearance error **{row['Mean absolute disappearance error (days)']:.2f} days**; "
        f"mean absolute ablation-duration error **{row['Mean absolute ablation-duration error (days)']:.2f} days**.")
display(Markdown("\n".join(lines)))


,Mean seasonal RMSE (cm),Mean signed bias (cm),Mean absolute seasonal bias (cm),Mean absolute peak-depth error (cm),Mean absolute disappearance error (days),Mean absolute ablation-duration error (days)
Forcing,,,,,,
HRRR-AK,28.51,14.15,15.39,31.69,9.00,11.11
Met Mixed,31.41,2.98,18.57,38.27,8.89,31.22


,Paired site–years used
Mean seasonal RMSE (cm),9
Mean signed bias (cm),9
Mean absolute seasonal bias (cm),9
Mean absolute peak-depth error (cm),9
Mean absolute disappearance error (days),9
Mean absolute ablation-duration error (days),9


- Average seasonal snow-depth RMSE: **HRRR-AK 28.51 cm**; **Met Mixed 31.41 cm**.
- Met Mixed − HRRR-AK average RMSE: **+2.90 cm (+10.2%)**.
- Met Mixed has lower RMSE in **4/9 site–years**; HRRR-AK in **5/9**; **0 ties**.
- **HRRR-AK:** mean signed bias **+14.15 cm**; mean absolute peak-depth error **31.69 cm**; mean absolute disappearance error **9.00 days**; mean absolute ablation-duration error **11.11 days**.
- **Met Mixed:** mean signed bias **+2.98 cm**; mean absolute peak-depth error **38.27 cm**; mean absolute disappearance error **8.89 days**; mean absolute ablation-duration error **31.22 days**.